# Laboratory 00 — Phasors: the language of waves

In this laboratory you will add oscillations by adding arrows — and use the arrows to
predict silence, beats, and the difference between a laser and a lamp.

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell. That is not a ritual: a prediction you
have committed to is the only reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | harmonic oscillations of one angular frequency, each represented by a complex amplitude (a phasor) |
| **Dynamics** | uniform rotation only — every phasor turns as $e^{-i\omega t}$; nothing else happens |
| **Boundary** | none; phasors neither grow, decay, nor exchange anything |
| **Ensemble** | deterministic, except the random-phase experiment: independent uniform draws, seeded |
| **Ignored** | everything physical — what oscillates, and every mechanism that would change an amplitude in time |
| **Valid when** | the signals really are single-frequency sinusoids |
| **Failure modes** | different frequencies summed as if phasors; nonlinear operations applied to the complex signal |

All the physics lives in `wavelab.phasors` — open it and read it; it is shorter than this
notebook.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Those builds still have to be asked for by name:
# the kernel fetches a Pyodide package by itself only when a cell's own text imports it, and
# scipy is imported inside wavelab, where it does not look. Add any new compiled dependency
# of wavelab to the first list below and any new *pure-Python* one to the second.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["numpy", "scipy"])
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import phasors
from wavelab.constants import SIGN_CONVENTION
from wavelab.validation import scaling_exponent

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

OMEGA = 2 * np.pi  # rad/s — period 1 s, so time axes read directly in cycles
print(SIGN_CONVENTION)

## Part 1 — One rotating arrow

The oscillation $x(t) = A\cos(\omega t + \varphi)$ is the horizontal shadow of an arrow
of length $A$ rotating **clockwise**: $x(t) = \mathrm{Re}[\hat{x}\, e^{-i\omega t}]$,
with the phasor $\hat{x} = A e^{-i\varphi}$ fixed and the rotation shared by every
oscillation of that frequency.

Run the next cell to see one arrow and its shadow.

In [ ]:
amplitude, phase = 1.0, np.pi / 3
z = phasors.phasor(amplitude, phase)
t = np.linspace(0.0, 2.0, 400)
signal = phasors.real_signal(amplitude, OMEGA, phase, t)

fig, (left, right) = plt.subplots(
    1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.8]}
)
snapshots = phasors.evaluate(z, OMEGA, np.arange(8) / 8.0)
left.axhline(0, color="0.85"), left.axvline(0, color="0.85")
for i, w in enumerate(snapshots):
    left.annotate(
        "", xy=(w.real, w.imag), xytext=(0, 0),
        arrowprops={"arrowstyle": "->", "color": plt.cm.viridis(i / 8)},
    )
left.add_patch(plt.Circle((0, 0), amplitude, fill=False, color="0.9"))
left.set_xlim(-1.2, 1.2), left.set_ylim(-1.2, 1.2), left.set_aspect("equal")
left.set_xlabel("Re"), left.set_ylabel("Im")
left.set_title("one turn in eight snapshots — clockwise")
right.plot(t, signal, lw=1.5)
right.set_xlabel("t (s)"), right.set_ylabel("x(t)")
right.set_title("the shadow: a cosine with the phasor's amplitude and phase")
plt.tight_layout()
plt.show()

## Part 2 — Add two oscillations

### Predict

Two oscillations of equal amplitude $a$ and the *same* frequency are added.

1. At phase difference $\delta = \pi$: what is the resultant amplitude?
2. At $\delta = \pi/2$?
3. Sketch the resultant amplitude as a function of $\delta$ from $0$ to $2\pi$ — where
   are its maximum and its zeros?

Write your prediction **before** running the next cell.

**Your prediction:**

In [ ]:
a = 1.0
deltas = np.linspace(0.0, 2 * np.pi, 200)
resultants = np.array([phasors.resultant([a, a], [0.0, d])[0] for d in deltas])

# The law of cosines, predicted analytically:
predicted = np.sqrt(2 * a**2 * (1.0 + np.cos(deltas)))

# And the honest brute force: sample each summed signal in time and take its peak.
t_fine = np.linspace(0.0, 1.0, 2001)
brute = np.array(
    [np.max(np.abs(a * np.cos(OMEGA * t_fine) + a * np.cos(OMEGA * t_fine + d))) for d in deltas]
)

plt.figure(figsize=(7, 4))
plt.plot(deltas, resultants, lw=2, label="phasor sum |z1 + z2|")
plt.plot(deltas, predicted, "--", lw=1.2, label="law of cosines")
plt.plot(deltas, brute, ":", lw=1.6, label="brute-force peak of the summed signal")
plt.xlabel("phase difference δ (rad)"), plt.ylabel("resultant amplitude")
plt.legend(), plt.tight_layout()
plt.show()

Three independent computations, one curve. The sum of two equal amplitudes runs from $2a$
(in phase) through $\sqrt{2}\,a$ (quarter cycle) to exactly **zero** (opposed): adding a
signal can silence a signal. This curve *is* noise cancellation — and in module 23 it will
be the interference pattern of two slits.

## Part 3 — Beats

Detune the second oscillation slightly. Now there is no common rotating frame to divide
out, and the sum is not a cosine at all: the two arrows drift through agreement and
opposition at the *difference* frequency.

### Predict

440 Hz plus 441 Hz, equal amplitudes: how many loudness pulses per second? And 440 Hz plus
444 Hz? Write it down, then run the cell (slowed a hundredfold so you can see it).

**Your prediction:**

In [ ]:
f1, f2 = 4.0, 5.0  # Hz — think "440 and 550", slowed a hundredfold to be visible
t_long = np.linspace(0.0, 3.0, 3001)
signal = phasors.beat_signal(1.0, 1.0, 2 * np.pi * f1, 2 * np.pi * f2, t_long)
envelope = 2.0 * np.cos(np.pi * (f1 - f2) * t_long)

plt.figure(figsize=(9, 3.6))
plt.plot(t_long, signal, lw=0.9)
plt.plot(t_long, np.abs(envelope), "--", color="0.5", lw=1.2)
plt.plot(t_long, -np.abs(envelope), "--", color="0.5", lw=1.2)
plt.xlabel("t (s)"), plt.ylabel("summed signal")
plt.title("4 Hz + 5 Hz: a 4.5 Hz carrier, loudness pulsing once per second")
plt.tight_layout()
plt.show()

# Measure the beat period as the spacing of the envelope's zeros, and compare:
crossings = np.nonzero(np.diff(np.signbit(envelope)))[0]
beat_period = float(np.mean(np.diff(t_long[crossings])))
print(f"measured beat period {beat_period:.3f} s   vs   1/|f1-f2| = {1.0 / abs(f1 - f2):.3f} s")

## Part 4 — One hundred violinists

$N$ unit phasors with independent random phases: how long is the sum arrow, typically?
Re-run the cell — the resultant differs every time, but its *typical* size obeys a law, and
the law separates a lamp from a laser.

In [ ]:
sizes = np.array([10, 100, 1000])
n_trials = 200
rms = []
for n in sizes:
    magnitudes = [abs(phasors.random_phasor_sum(int(n), rng)) for _ in range(n_trials)]
    rms.append(float(np.sqrt(np.mean(np.square(magnitudes)))))

exponent = scaling_exponent(sizes, rms)
plt.figure(figsize=(6, 4))
plt.loglog(sizes, rms, "o-", label=f"RMS resultant (fitted slope {exponent:.2f})")
plt.loglog(sizes, np.sqrt(sizes), "--", color="0.5", label="sqrt(N) — random walk")
plt.loglog(sizes, sizes, ":", color="0.7", label="N — phase-locked")
plt.xlabel("number of phasors N"), plt.ylabel("resultant amplitude")
plt.legend(), plt.tight_layout()
plt.show()
print(f"fitted exponent {exponent:.3f}   (random phases: 1/2; locked phases would give 1)")
print(f"intensity, locked vs random, at N=100: {100**2 // 100}x — the case for the laser")

## Part 5 — Automated checks

The claims this notebook has made, as executable assertions — the same checks run in the
project's test suite, so the page, the notebook and the library cannot silently drift
apart.

In [ ]:
# 1. The representation: the phasor round trip reproduces the direct cosine exactly.
t_check = np.linspace(0.0, 5.0, 2001)
via_phasor = phasors.real_signal(0.7, 3.0, 1.1, t_check)
assert np.max(np.abs(via_phasor - 0.7 * np.cos(3.0 * t_check + 1.1))) < 1e-12

# 2. The interference law, at an arbitrary phase difference.
amp, _ = phasors.resultant([0.8, 1.3], [0.0, 0.9])
assert abs(amp - np.sqrt(0.8**2 + 1.3**2 + 2 * 0.8 * 1.3 * np.cos(0.9))) < 1e-12

# 3. Complete cancellation and complete reinforcement.
cancelled, _ = phasors.resultant([1.0, 1.0], [0.0, np.pi])
doubled, _ = phasors.resultant([1.0, 1.0], [0.3, 0.3])
assert cancelled < 1e-12 and abs(doubled - 2.0) < 1e-12

# 4. The beat identity: the honest sum equals carrier times envelope.
w1, w2 = 3.0, 3.4
beat = phasors.beat_signal(0.6, 0.6, w1, w2, t_check)
identity = 2 * 0.6 * np.cos(0.5 * (w1 - w2) * t_check) * np.cos(0.5 * (w1 + w2) * t_check)
assert np.max(np.abs(beat - identity)) < 1e-12

# 5. The random walk: fitted exponent within 0.1 of 1/2, on fresh seeds.
check_rng = np.random.default_rng(1)
check_rms = [
    float(np.sqrt(np.mean([abs(phasors.random_phasor_sum(n, check_rng)) ** 2 for _ in range(100)])))
    for n in (10, 100, 1000)
]
assert abs(scaling_exponent([10, 100, 1000], check_rms) - 0.5) < 0.1

print("all checks passed")

## Part 6 — Explore it yourself

Two experiments worth doing:

1. Find an amplitude pair ($A_1 \neq A_2$) and a phase difference at which the resultant
   equals $A_1$ exactly — a second oscillation present yet invisible in the amplitude.
2. Set $A_2 = 0.99\,A_1$ and $\delta = 180°$. How small is the residual? That is how well
   noise cancellation must match amplitudes to earn its name.

In [ ]:
import ipywidgets as widgets


def explore(a1=1.0, a2=0.7, delta_degrees=120.0):
    delta = np.radians(delta_degrees)
    z1, z2 = phasors.phasor(a1, 0.0), phasors.phasor(a2, delta)
    amp, phi = phasors.resultant([a1, a2], [0.0, delta])
    tt = np.linspace(0.0, 2.0, 800)
    reach = max(a1 + a2, 0.1)

    fig, (left, right) = plt.subplots(
        1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.8]}
    )
    left.axhline(0, color="0.85"), left.axvline(0, color="0.85")
    left.annotate("", xy=(z1.real, z1.imag), xytext=(0, 0),
                  arrowprops={"arrowstyle": "->", "color": "tab:blue"})
    left.annotate("", xy=(z1.real + z2.real, z1.imag + z2.imag), xytext=(z1.real, z1.imag),
                  arrowprops={"arrowstyle": "->", "color": "tab:orange"})
    left.annotate("", xy=(z1.real + z2.real, z1.imag + z2.imag), xytext=(0, 0),
                  arrowprops={"arrowstyle": "->", "color": "tab:green", "lw": 2})
    left.set_xlim(-reach, reach), left.set_ylim(-reach, reach), left.set_aspect("equal")
    left.set_xlabel("Re"), left.set_ylabel("Im")
    left.set_title(f"resultant amplitude {amp:.3f}")
    right.plot(tt, phasors.real_signal(a1, OMEGA, 0.0, tt), lw=0.9, color="tab:blue")
    right.plot(tt, phasors.real_signal(a2, OMEGA, delta, tt), lw=0.9, color="tab:orange")
    right.plot(tt, phasors.real_signal(amp, OMEGA, phi, tt), lw=1.8, color="tab:green")
    right.set_xlabel("t (s)"), right.set_ylim(-reach, reach)
    plt.tight_layout()
    plt.show()


widgets.interact(
    explore,
    a1=widgets.FloatSlider(min=0.0, max=2.0, step=0.05, value=1.0, description="A1"),
    a2=widgets.FloatSlider(min=0.0, max=2.0, step=0.05, value=0.7, description="A2"),
    delta_degrees=widgets.FloatSlider(min=0.0, max=360.0, step=5.0, value=120.0,
                                      description="δ (deg)"),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "00-phasors.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. This model represents each oscillation by a *fixed* complex amplitude. Name the moment
   in this notebook where that representation broke down, and what replaced it.
3. Explain, without equations, why one hundred random-phase sources are one hundred times
   one source, while one hundred phase-locked sources are ten thousand times.

**Your answers:**

1.
2.
3.